In [ ]:
# ── Cell 1: Pull latest code from GitHub ─────────────────────────────────────
# Run every session to pick up any new pushes.
import sys, os, subprocess

REPO_URL = 'https://github.com/moizaimran/curesense-project.git'
BRANCH   = 'hassan-branch'
REPO_DIR = '/kaggle/working/curesense-project'

if os.path.exists(REPO_DIR):
    print('Pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

sys.path.insert(0, REPO_DIR)
print('Ready:', REPO_DIR)

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
import sys
REPO_DIR = '/kaggle/working/curesense-project'
!pip install -r {REPO_DIR}/requirements.txt -q
!pip install pyngrok rapidfuzz torchvision --quiet

In [ ]:
# ── Cell 3: Secrets + OpenAI client ──────────────────────────────────────────
# Kaggle secrets required (Settings → Secrets):
#   'Ngrok Key'  — ngrok auth token
#   'OpenAI Key' — OpenAI API key
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
from openai import OpenAI
import glinker.config as cfg

_secrets = UserSecretsClient()
ngrok.set_auth_token(_secrets.get_secret('Ngrok Key'))
cfg.openai_client = OpenAI(api_key=_secrets.get_secret('OpenAI Key'))

print('OpenAI client ready')
print('ngrok ready')

In [ ]:
# ── Cell 4: Load Whisper + GLiNER on GPU ────────────────────────────────────
import torch, whisper
from gliner import GLiNER
import glinker.config as cfg

print('Loading Whisper Large ...')
cfg.whisper_model = whisper.load_model('large', device='cuda')
print('Whisper ready')

print('Loading GLiNER BioMed ...')
cfg.gliner_model = GLiNER.from_pretrained('Ihor/gliner-biomed-bi-large-v1.0').to('cuda')
print('GLiNER ready on:', next(cfg.gliner_model.parameters()).device)

In [ ]:
# ── Cell 5: Load dual RAG index (clinician + patient) ─────────────────────────
#
# Priority order (fastest first):
#   1. /kaggle/working/rag_index/          — already in this session → skip copy
#   2. /kaggle/input/uresense-rag-index/   — pre-built Kaggle dataset → copy once
#   3. Build from scratch                  — ~60-90 min, first time only
#
# Dataset contains 4 files:
#   clinician_index.faiss + clinician_meta.json  (textbooks + clinician guidelines)
#   patient_index.faiss   + patient_meta.json    (wikidoc patient corpus)
#
import os, shutil, glob
import glinker.config as cfg

WORKING = cfg.RAG_INDEX_DIR   # /kaggle/working/rag_index
RAG_FILES = (
    'clinician_index.faiss', 'clinician_meta.json',
    'patient_index.faiss',   'patient_meta.json',
)

if os.path.exists(f'{WORKING}/clinician_index.faiss'):
    print('RAG index already in working dir — Cell 7 will load it.')
else:
    # Search both known Kaggle dataset mount paths
    candidate = None
    for root in [
        '/kaggle/input/uresense-rag-index',
        '/kaggle/input/datasets/hassanraheem/uresense-rag-index',
    ]:
        hits = (glob.glob(f'{root}/**/clinician_index.faiss', recursive=True)
                or glob.glob(f'{root}/clinician_index.faiss'))
        if hits:
            candidate = os.path.dirname(hits[0])
            break

    if candidate:
        print(f'Pre-built dataset found at: {candidate}')
        os.makedirs(WORKING, exist_ok=True)
        for fname in RAG_FILES:
            src = f'{candidate}/{fname}'
            if os.path.exists(src):
                print(f'  Copying {fname} ...')
                shutil.copy(src, f'{WORKING}/{fname}')
            else:
                print(f'  WARNING: {fname} not in dataset — that index will degrade gracefully')
        print('Copy complete. Cell 7 will load into memory.')
    else:
        print('Dataset not mounted — building from scratch (~60-90 min) ...')
        print('Add uresense-rag-index dataset to this notebook to skip the build next time.')
        from glinker.rag.ingestion import build_index
        build_index(textbooks_limit=60_000)  # guidelines + wikidoc: full corpus (no cap)
        print('Build complete. Cell 7 will load into memory.')

In [ ]:
# ── Cell 6: Load disease index ────────────────────────────────────────────────
#
# Priority order:
#   1. /kaggle/working/disease_index/          — already in this session → skip
#   2. /kaggle/input/curesense-disease-index/  — pre-built Kaggle dataset → copy
#   3. Build from scratch                      — ~5-10 min, first time only
#
import os, shutil, glob

WORKING_DISEASE = '/kaggle/working/disease_index'
DISEASE_FILES   = ('disease_index.faiss', 'disease_meta.json')

if os.path.exists(f'{WORKING_DISEASE}/disease_index.faiss'):
    print('Disease index already in working dir — Cell 7 will load it.')
else:
    candidate = None
    for root in [
        '/kaggle/input/curesense-disease-index',
        '/kaggle/input/datasets/hassanraheem/curesense-disease-index',
    ]:
        hits = (glob.glob(f'{root}/**/disease_index.faiss', recursive=True)
                or glob.glob(f'{root}/disease_index.faiss'))
        if hits:
            candidate = os.path.dirname(hits[0])
            break

    if candidate:
        print(f'Pre-built disease dataset found at: {candidate}')
        os.makedirs(WORKING_DISEASE, exist_ok=True)
        for fname in DISEASE_FILES:
            print(f'  Copying {fname} ...')
            shutil.copy(f'{candidate}/{fname}', f'{WORKING_DISEASE}/{fname}')
        print('Copy complete. Cell 7 will load into memory.')
    else:
        print('Dataset not mounted — building from scratch (~5-10 min) ...')
        print('Add curesense-disease-index dataset to this notebook to skip next time.')
        from scripts.build_disease_index import build_disease_index
        build_disease_index()
        print('Build complete. Cell 7 will load into memory.')

In [ ]:
# ── Cell 7: Load indexes into memory ─────────────────────────────────────────
# Must run AFTER Cells 5 & 6 and BEFORE Cell 8.
# Flask serves requests the moment it starts — skipping this cell means
# all RAG and disease lookups silently return [] for the whole session.
from glinker.rag.retrieval import load_index
from glinker.disease.disease_retrieval import load_disease_index

load_index()          # clinician_index.faiss + patient_index.faiss → memory
load_disease_index()  # disease_index.faiss → memory

In [ ]:
# ── Cell 8: Start Flask on port 5001 ─────────────────────────────────────────
import threading
from api.app import app as flask_app

_flask_thread = threading.Thread(
    target=lambda: flask_app.run(port=5001, debug=False, use_reloader=False),
    daemon=True,
)
_flask_thread.start()
print('Flask started on port 5001')

In [ ]:
# ── Cell 9: ngrok tunnel → Flask:5001 ────────────────────────────────────────
# Exposes Flask at a public HTTPS URL → use as AI_SERVICE_URL in Backend/.env.
# MedGemma has its own URL from the Colab notebook → MEDGEMMA_SERVICE_URL.
import time, socket
from pyngrok import ngrok

def _port_up(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

for _ in range(15):
    if _port_up(5001): break
    time.sleep(1)

for t in ngrok.get_tunnels():
    print(f'[ngrok] Closing old tunnel: {t.public_url}')
    ngrok.disconnect(t.public_url)

PUBLIC_URL = ngrok.connect(5001).public_url

print(f'\n{"="*64}')
print(f'  AI_SERVICE_URL = {PUBLIC_URL}')
print(f'{"="*64}')
print('\nPaste into Backend/.env as AI_SERVICE_URL and restart Express.')
print('Also paste the Colab URL as MEDGEMMA_SERVICE_URL.')